In [1]:
import numpy as np
import pandas as pd

In [89]:
nz_results=pd.DataFrame(columns=["Z score",'Win percentage','Tie percentage','Home Matches','Home Win Percentage','Away win Percentage','Sport'])

# Start of Cricket Analysis

In [4]:
# Cleaning cricket Dataset
cricket_df=pd.read_csv("icc_cwc.csv")
cricket_df=cricket_df.dropna()
# Dropping rows not neccessary for analysis
cricket_df=cricket_df.drop(["Inns","Opposition Innings","Overs","Opposition Overs"],axis=1)
# Cleaning so Opposition is just country name
cricket_df["Opposition"]=cricket_df["Opposition"].apply(lambda x: x.replace('v','').strip())
# Converting score to integer for Score and Opposition score
cricket_df["Score"]=pd.to_numeric(cricket_df["Score"].apply(lambda x: x.split("/")[0]),errors='coerce')
cricket_df["Opposition Score"]=pd.to_numeric(cricket_df["Opposition Score"].apply(lambda x: x.split("/")[0]))

In [5]:
# Make list of NZ cities so can establish when game is played on Home ground i.e. in New Zealand
nz_cricket_cities=['Auckland','Hamilton','Napier','Dunedin','Wellington','Christchurch']

In [6]:
# Getting all matches where NZ is Team 1 or Opposition
nz=cricket_df[(cricket_df['Team 1']=="New Zealand")|(cricket_df["Opposition"]=="New Zealand")]
# Creating new Column Home to show when game is played in NZ
nz["Home"]=nz["Ground"].isin(nz_cricket_cities)
nz_team1=nz[nz["Team 1"]=="New Zealand"]
nz_opp=nz[nz["Opposition"]=="New Zealand"]
# Number of matches where NZ as Team 1 Won
nz_win=len(nz[(nz["Team 1"]=="New Zealand")&(nz["Result"]=="won")])
# Number of matches where NZ as opposition won. Thus meaning Team 1 lost.
other_lost=len(nz[(nz["Opposition"]=="New Zealand")&(nz["Result"]=='lost')])

# Win Percentage as total number of matches won divided by total matches that NZ played
nz_win_percent=round((nz_win+other_lost)/(len(nz)),3)
# Percentage of matches where NZ tied
nz_tie_percent=round(len(nz[nz['Result']=="tied"])/len(nz),2)

In [7]:
# Computing mean score for NZ cricket scores
nz_cricket_df=np.concatenate((nz_team1["Score"],nz_opp["Opposition Score"]))
nz_cricket_mean=nz_cricket_df.mean()
nz_cricket_std=nz_cricket_df.std()**2/len(nz_cricket_df)

In [8]:
# Mean score for all teams in Dataset
total_cricket_df=pd.DataFrame(np.concatenate((cricket_df["Score"],cricket_df["Opposition Score"])))
total_cricket_mean=total_cricket_df.mean()
total_cricket_std=total_cricket_df.std()**2/len(total_cricket_df)

In [36]:
# Z score for testing significance of result
cricket_z_score=pd.to_numeric((nz_cricket_mean-total_cricket_mean)/(np.sqrt(nz_cricket_std+total_cricket_std)),errors="coerce")[0]

In [37]:
cricket_z_score

np.float64(0.3913623792568213)

In [10]:
nz_cricket_away=nz[nz['Home']==False]
nz_win_away=len(nz_cricket_away[(nz_cricket_away["Team 1"]=="New Zealand")&(nz_cricket_away["Result"]=="won")])
other_lost_away=len(nz_cricket_away[(nz_cricket_away["Opposition"]=="New Zealand")&(nz_cricket_away["Result"]=='lost')])
nz_win_percent_away=round((nz_win_away+other_lost_away)/(len(nz_cricket_away)),3)

In [13]:
nz_cricket_home=nz[nz["Home"]==True]
nz_win_home=len(nz_cricket_home[(nz_cricket_home["Team 1"]=="New Zealand")&(nz_cricket_home["Result"]=="won")])
other_lost_home=len(nz_cricket_home[(nz_cricket_home["Opposition"]=="New Zealand")&(nz_cricket_home["Result"]=='lost')])
nz_win_percent_home=round((nz_win_home)/(len(nz_cricket_home)),3)

In [39]:
nz_results.loc[0]=[cricket_z_score,nz_win_percent,nz_tie_percent,len(nz_cricket_home),nz_win_percent_home,nz_win_percent_away,'Cricket']

# Start of Rugby Analysis

In [42]:
rugby_df=pd.read_csv('rugby_results.csv')
rugby_df=rugby_df.drop(['competition','stadium','city','neutral'],axis=1)
nz_rugby=rugby_df[(rugby_df['home_team']=="New Zealand")|(rugby_df['away_team']=="New Zealand")]

In [43]:
nz_away_win=nz_rugby[(nz_rugby["away_team"]=="New Zealand")&(nz_rugby["away_score"]>nz_rugby["home_score"])]
nz_home_win=nz_rugby[(nz_rugby['home_team']=="New Zealand")&(nz_rugby['home_score']>nz_rugby['away_score'])]
nz_win_percent=round((len(nz_away_win)+len(nz_home_win))/len(nz_rugby),3)

In [44]:
nz_home_rugby=nz_rugby[nz_rugby['country']=="New Zealand"]
nz_win_home=round(len(nz_home_rugby[nz_home_rugby["home_score"]>nz_home_rugby['away_score']])/(len(nz_home_rugby)),2)

In [45]:
nz_rugby_score=np.concatenate((nz_rugby[nz_rugby["home_team"]=="New Zealand"]["home_score"],nz_rugby[nz_rugby["away_team"]=="New Zealand"]['away_score']))
nz_mean_rugby=nz_rugby_score.mean()
nz_std_rugby=nz_rugby_score.std()**2/len(nz_rugby)

In [46]:
all_score_rugby=np.concatenate((rugby_df["home_score"],rugby_df["away_score"]))
all_mean_rugby=all_score_rugby.mean()
all_std_rugby=all_score_rugby.std()**2/len(all_score_rugby)

In [47]:
rugby_z_score=(nz_mean_rugby-all_mean_rugby)/(np.sqrt(nz_std_rugby+all_std_rugby))

In [49]:
tied_rugby=round(len(nz_rugby[nz_rugby['home_score']==nz_rugby['away_score']])/len(nz_rugby),3)

In [51]:
nz_away=nz_rugby[nz_rugby["country"]!='New Zealand']
a=len(nz_away)
b=len(nz_away[(nz_away["home_team"]=="New Zealand")&(nz_away["home_score"]>nz_away['away_score'])])
c=len(nz_away[(nz_away["away_team"]=="New Zealand")&(nz_away["away_score"]>nz_away['home_score'])])
nz_away_win=round((b+c)/a,3)

In [52]:
nz_results.loc[1]=[rugby_z_score,nz_win_percent,tied_rugby,len(nz_home_rugby),nz_win_home,nz_away_win,'Rugby']

In [53]:
nz_results

,Mean score,Win percentage,Tie percentage,Home Matches,Home Win Percentage,Away win Percentage,Sport
0,0.391362,0.596,0.010,17,0.235,0.537,Cricket
1,13.098028,0.754,0.034,233,0.830,0.699,Rugby


# Start of football analysis

In [54]:
football_df=pd.read_csv("football_results.csv")
football_df=football_df.drop(['city','neutral'],axis=1)

In [55]:
nz_football=football_df[(football_df["home_team"]=="New Zealand")|(football_df["away_team"]=="New Zealand")]

In [56]:
nz_home_football=nz_football[nz_football['country']=="New Zealand"]

In [57]:
nz_football=nz_football.dropna()

In [58]:
# Win percentage when NZ team plays in New Zealand
nz_home_win=round(len(nz_home_football[nz_home_football["home_score"]>nz_home_football["away_score"]])/len(nz_home_football),3)

In [59]:
nz_home_football[nz_home_football["home_score"]>nz_home_football["away_score"]]

,date,home_team,away_team,home_score,away_score,tournament,country
737,1922-06-17,New Zealand,Australia,3.0,1.0,Soccer Ashes,New Zealand
741,1922-07-08,New Zealand,Australia,3.0,1.0,Soccer Ashes,New Zealand
1200,1927-07-09,New Zealand,Canada,1.0,0.0,Friendly,New Zealand
5508,1962-06-02,New Zealand,New Caledonia,4.0,1.0,Friendly,New Zealand
5515,1962-06-04,New Zealand,New Caledonia,4.0,2.0,Friendly,New Zealand
...,...,...,...,...,...,...,...
47882,2024-11-15,New Zealand,Vanuatu,8.0,1.0,FIFA World Cup qualification,New Zealand
47945,2024-11-18,New Zealand,Samoa,8.0,0.0,FIFA World Cup qualification,New Zealand
48134,2025-03-21,New Zealand,Fiji,7.0,0.0,FIFA World Cup qualification,New Zealand
48178,2025-03-24,New Zealand,New Caledonia,3.0,0.0,FIFA World Cup qualification,New Zealand


In [61]:
nz_home_notnz_win=nz_football[(nz_football["country"]!="New Zealand")&(nz_football["home_team"]=="New Zealand")&(nz_football["home_score"]>nz_football["away_score"])]
nz_away_win=nz_football[(nz_football["away_team"]=="New Zealand")&(nz_football["away_score"]>nz_football["home_score"])]
nz_away_win_percent=round((len(nz_home_notnz_win)+len(nz_away_win))/len(nz_football[nz_football["country"]!="New Zealand"]),3)

In [62]:
nz_win_percent=round((nz_home_win+nz_away_win_percent)/2,3)

In [64]:
# Computing mean score for NZ team
nz_football_home=nz_football[nz_football["home_team"]=="New Zealand"]['home_score'].mean()
nz_away=nz_football[nz_football["away_team"]=="New Zealand"]["away_score"].mean()
nz_football_points=round((nz_football_home+nz_away)/2,3)

In [65]:
nz_football_score_df=np.concatenate((nz_football[nz_football["home_team"]=="New Zealand"]["home_score"],nz_football[nz_football["away_team"]=="New Zealand"]["away_score"]))
nz_football_mean=nz_football_score_df.mean()
nz_football_std=nz_football_score_df.std()**2/len(nz_football_score_df)

In [66]:
all_football_df=pd.DataFrame(np.concatenate((football_df["home_score"],football_df["away_score"])))
all_football_mean=all_football_df.mean()
all_football_std=all_football_df.std()**2/len(all_football_df)

In [83]:
football_z_score=np.round((nz_football_mean-all_football_mean)/(np.sqrt(nz_football_std+all_football_std))[0],3)

In [84]:
football_z_score

0    2.963
dtype: float64

In [80]:
# Percentage of games that NZ football team ties
nz_tied_football=round((len(nz_football[nz_football['home_score']==nz_football["away_score"]]))/(len(nz_football)),3)

In [85]:
# Adding NZ records to results dataframe. [mean points, win percentage, tied percentage,Home matches,home win percent, away win percent ]
nz_results.loc[2]=[football_z_score[0],nz_win_percent,nz_tied_football,len(nz_home_football),nz_home_win,nz_away_win_percent,"Football"]

In [157]:
# Get all FIFA related records
nz_fifa=nz_football[nz_football["tournament"].str.contains("FIFA",regex=True)]

In [119]:
nz_results.to_csv("nz_results.csv")

In [87]:
nz_results["Mean score"]

0     0.391362
1    13.098028
2        2.963
Name: Mean score, dtype: object

In [ ]:
# Standardised score across all sports. 

In [88]:
nz_results

,Mean score,Win percentage,Tie percentage,Home Matches,Home Win Percentage,Away win Percentage,Sport
0,0.391362,0.596,0.010,17,0.235,0.537,Cricket
1,13.098028,0.754,0.034,233,0.830,0.699,Rugby
2,2.963,0.458,0.176,128,0.570,0.346,Football


In [ ]:
# Analysis from all sports countries we lost to most